# Análise Envoltória de Dados (DEA) na prática com Python

Este notebook tem como objetivo apresentar o uso da Análise Envoltória de Dados (DEA) em Python, utilizando a biblioteca `adndea` para calcular a eficiência de Unidades Tomadoras de Decisão (DMUs).

## 1. Importando a biblioteca `adndea` com Git

Primeiramente, precisamos importar a biblioteca `adndea` para o ambiente do Colab. Esta biblioteca está disponível em um repositório Git e será clonada diretamente para o nosso ambiente de execução.

In [1]:
!git clone https://gitlab.com/aloisio/adndea.git

Cloning into 'adndea'...
remote: Enumerating objects: 36, done.
remote: Counting objects: 100% (6/6), done.
remote: Compressing objects: 100% (6/6), done.
remote: Total 36 (delta 2), reused 0 (delta 0), pack-reused 30 (from 1)
Receiving objects: 100% (36/36), 7.04 KiB | 7.04 MiB/s, done.
Resolving deltas: 100% (15/15), done.


## 2. Montando o Google Drive

Para acessar os arquivos de dados necessários para este laboratório, montaremos o Google Drive. Isso nos permitirá ler e escrever arquivos de e para o seu Drive.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
DATA_PATH = '/content/drive/MyDrive/Aulas/Aulas IDP/2026/Auditoria de Dados e Accountability/modulo III - Aud operacional/dea'

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 3. Importando os módulos necessários

Vamos importar as bibliotecas Python que utilizaremos:
- `pandas` para manipulação e análise de dados.
- `os` para interagir com o sistema operacional, especialmente para lidar com caminhos de arquivo.
- `numpy` para operações numéricas e criação de arrays.
- `adndea.solver` que contém as funções dos modelos DEA (CCR e BCC, orientados a input/output).

In [ ]:
import pandas as pd
import os
import numpy as np
import adndea.solver as dea

## 4. Lendo o primeiro conjunto de dados (`dea_data_1.csv`)

Vamos carregar o primeiro conjunto de dados, `dea_data_1.csv`, em um DataFrame do pandas. Este dataset contém informações sobre as Unidades Tomadoras de Decisão (DMUs) e seus respectivos inputs (entradas) e outputs (saídas) para a análise DEA.

In [ ]:
df = pd.read_csv(os.path.join(DATA_PATH, "dea_data_1.csv"))
df

,dmu,in1,ou1,ou2,ou3
0,dmu1,250,16,12,4.0
1,dmu2,225,16,8,5.0
2,dmu3,300,32,16,4.5
3,dmu4,275,32,8,4.0


## 5. Preparando as Matrizes de Input (X) e Output (Y)

Para a análise DEA, precisamos separar os dados em duas matrizes: uma para os **inputs (entradas)** e outra para os **outputs (saídas)**. Neste primeiro conjunto de dados:
- `in1` representa o input.
- `ou1`, `ou2`, `ou3` representam os outputs.

In [ ]:
X = np.array(df[['in1']])
Y = np.array(df[['ou1','ou2','ou3']])

In [ ]:
X

array([[250],
       [225],
       [300],
       [275]])

In [ ]:

Y

array([[16. , 12. ,  4. ],
       [16. ,  8. ,  5. ],
       [32. , 16. ,  4.5],
       [32. ,  8. ,  4. ]])

## 6. Executando o Solver CCR Orientado a Output (`CCR_O`)

O modelo CCR (Charnes, Cooper, Rhodes) assume retornos constantes de escala. A orientação a output (`_O`) significa que estamos maximizando os outputs para um dado nível de inputs. O solver nos retornará os pesos ótimos para cada input/output e a eficiência relativa de cada DMU.

In [ ]:
resp = dea.CCR_O(X, Y)
resp

[(array([0.     , 0.04577, 0.11268, 0.00413]), 0.96818),
 (array([0.     , 0.     , 0.2    , 0.00444]), 1.0),
 (array([0.     , 0.0625 , 0.     , 0.00333]), 1.0),
 (array([0.03125, 0.     , 0.     , 0.00364]), 1.0)]

## 7. Apresentando os resultados do modelo CCR_O

Vamos exibir os resultados para cada DMU, mostrando os pesos atribuídos aos inputs e outputs, e a eficiência relativa calculada pelo modelo CCR orientado a output. A eficiência relativa varia de 0 a 1, onde 1 indica uma DMU eficiente.

In [ ]:
for dmu, r in zip(df.dmu,resp):
    print("{:s} - pesos:{} - Ef. Relativa: {:.5f}".format(dmu, r[0], r[1]))

dmu1 - pesos:[0.      0.04577 0.11268 0.00413] - Ef. Relativa: 0.96818
dmu2 - pesos:[0.      0.      0.2     0.00444] - Ef. Relativa: 1.00000
dmu3 - pesos:[0.      0.0625  0.      0.00333] - Ef. Relativa: 1.00000
dmu4 - pesos:[0.03125 0.      0.      0.00364] - Ef. Relativa: 1.00000


## 8. Comparando com o modelo BCC Orientado a Output (`BCC_O`)

O modelo BCC (Banker, Charnes, Cooper) assume retornos variáveis de escala, sendo uma extensão do modelo CCR. A orientação a output (`_O`) também significa que estamos maximizando os outputs para um dado nível de inputs. Vamos executar este solver para comparar os resultados com o modelo CCR.

In [ ]:
resp = dea.BCC_O(X, Y)
resp

[(array([0.     , 0.04577, 0.11268, 0.00413, 0.     ]), 0.96818),
 (array([0. , 0. , 0.2, 0. , 1. ]), 1.0),
 (array([0.    , 0.0625, 0.    , 0.    , 1.    ]), 1.0),
 (array([ 0.03125,  0.     , -0.     ,  0.     ,  1.     ]), 1.0)]

In [ ]:
for dmu, r in zip(df.dmu,resp):
    print("{:s} - pesos:{} - Ef. Relativa: {:.5f}".format(dmu, r[0], r[1]))

dmu1 - pesos:[0.      0.04577 0.11268 0.00413 0.     ] - Ef. Relativa: 0.96818
dmu2 - pesos:[0.  0.  0.2 0.  1. ] - Ef. Relativa: 1.00000
dmu3 - pesos:[0.     0.0625 0.     0.     1.    ] - Ef. Relativa: 1.00000
dmu4 - pesos:[ 0.03125  0.      -0.       0.       1.     ] - Ef. Relativa: 1.00000


## Análise dos Resultados CCR_O e BCC_O para `dea_data_1.csv`

Ao analisar os resultados da eficiência das DMUs para o `dea_data_1.csv`, utilizando os modelos CCR (Constant Returns to Scale) e BCC (Variable Returns to Scale), ambos orientados a output, podemos observar o seguinte:

### Modelo CCR_O (Eficiências do Item 7):
*   **DMU1:** Apresentou uma eficiência relativa de 0.96818, indicando que não é totalmente eficiente sob retornos constantes de escala. Ela poderia aumentar seus outputs em aproximadamente 3.18% com os mesmos inputs, ou manter os outputs com 3.18% menos inputs, para se tornar eficiente.
*   **DMU2, DMU3, DMU4:** Todas apresentaram eficiência relativa de 1.00000, indicando que são eficientes sob retornos constantes de escala. Elas estão na fronteira de produção em relação ao conjunto de DMUs analisadas pelo modelo CCR.

### Modelo BCC_O (Eficiências do Item 8):
*   **DMU1, DMU2, DMU3, DMU4:** Todas as DMUs apresentaram eficiência relativa de 1.00000.

### Comparação e Conclusão para `dea_data_1.csv`:

Para este conjunto de dados (`dea_data_1.csv`), observamos uma diferença notável apenas para a DMU1. Enquanto no modelo CCR_O a DMU1 não é eficiente (0.96818), no modelo BCC_O ela atinge 1.00000 de eficiência. As DMU2, DMU3 e DMU4 são eficientes em ambos os modelos.

Esta diferença para a DMU1 é explicada pela suposição de retornos de escala de cada modelo:

*   **CCR_O:** Assume que as DMUs operam com retornos constantes de escala, ou seja, um aumento proporcional nos inputs leva a um aumento proporcional nos outputs. A fronteira de eficiência do CCR é uma linha (ou hiperplano) reta que envolve todas as DMUs.
*   **BCC_O:** Assume retornos variáveis de escala, o que significa que um aumento proporcional nos inputs pode levar a um aumento mais que proporcional, menos que proporcional ou proporcional nos outputs, dependendo da escala de operação da DMU. A fronteira de eficiência do BCC é 'envelopadora', formando uma fronteira côncava que se ajusta mais aos dados.

O fato de a DMU1 ser ineficiente no CCR_O mas eficiente no BCC_O sugere que sua ineficiência no modelo CCR pode ser atribuída à **ineficiência de escala**. Isso significa que, embora a DMU1 possa ser tecnicamente eficiente em sua escala de operação (conforme o BCC), ela não está operando na escala ótima que seria atingida se houvesse retornos constantes de escala (conforme o CCR).

### É possível que todas as unidades estejam na fronteira de eficiência com `dea_data_1.csv`?

Para este dataset específico, com apenas 4 DMUs e uma estrutura relativamente simples (1 input, 3 outputs), o modelo BCC_O classificou todas as DMUs como eficientes. No contexto do DEA, é *possível* que todas as unidades estejam na fronteira de eficiência, especialmente se o número de DMUs for pequeno ou se elas forem muito homogêneas em seu desempenho. Neste caso, o modelo BCC, com sua fronteira de eficiência mais flexível (retornos variáveis de escala), conseguiu 'envolver' todas as DMUs como eficientes.

Em situações com um número maior de DMUs e mais complexidade nos dados, é menos comum que todas as unidades sejam eficientes em ambos os modelos. A ocorrência de todas as DMUs como eficientes, especialmente no BCC, pode, por vezes, levantar a questão se o modelo está sendo muito permissivo ou se os dados realmente refletem essa alta eficiência geral.

## Exercício 1: Análise DEA com um novo conjunto de dados

Neste exercício, você aplicará os conceitos de DEA aprendidos para analisar um segundo conjunto de dados. Siga os passos abaixo:

*   **1. Ler o Segundo Arquivo de Dados (`dea_data_2.csv`)**
    *   Carregue o arquivo `dea_data_2.csv` em um DataFrame do pandas.
*   **2. Verifique a quantidade de entradas e saídas do arquivo**
    *   Analise o DataFrame para identificar quais colunas são inputs e quais são outputs.
*   **3. Separe as entradas e saídas em Matrizes distintas (X e Y)**
    *   Crie as matrizes `X` e `Y` usando `numpy.array`.
*   **4. Execute os dois modelos (CCR_O e BCC_O)**
    *   Aplique os solvers `dea.CCR_O` e `dea.BCC_O` nas novas matrizes.
*   **5. Apresente os resultados (Eficiência Relativa)**
    *   Exiba a eficiência relativa de cada DMU para ambos os modelos, de forma similar ao exemplo anterior.

## 9. Lendo o Segundo Arquivo de Dados (`dea_data_2.csv`)

Vamos agora carregar o arquivo `dea_data_2.csv`, que utilizaremos para o Exercício 1. Este dataset apresenta uma estrutura diferente em termos de número de inputs e outputs.

In [ ]:
df = pd.read_csv(os.path.join(DATA_PATH, "dea_data_2.csv"))
df

,dmu,in1,in2,in3,in4,ou1,ou2,ou3
0,dmu1,250,60,50,30,200,100,90
1,dmu2,1500,400,150,125,600,250,60
2,dmu3,800,350,300,85,600,450,40
3,dmu4,500,150,200,75,500,360,60
4,dmu5,200,100,120,60,330,250,50
5,dmu6,600,100,50,35,180,75,80
6,dmu7,1500,500,90,40,500,200,100
7,dmu8,1000,360,300,90,750,500,65
8,dmu9,530,120,100,60,350,180,50
9,dmu10,300,45,80,50,440,230,80


## 10. Separando Entradas (X) e Saídas (Y) para `dea_data_2.csv`
Para este novo conjunto de dados (`dea_data_2.csv`):
- As colunas `in1`, `in2`, `in3`, `in4` serão consideradas os inputs.
- As colunas `ou1`, `ou2`, `ou3` serão consideradas os outputs.

In [ ]:
X = np.array(df[['in1', 'in2', 'in3', 'in4']])
Y = np.array(df[['ou1','ou2','ou3']])

In [ ]:
X

array([[ 250,   60,   50,   30],
       [1500,  400,  150,  125],
       [ 800,  350,  300,   85],
       [ 500,  150,  200,   75],
       [ 200,  100,  120,   60],
       [ 600,  100,   50,   35],
       [1500,  500,   90,   40],
       [1000,  360,  300,   90],
       [ 530,  120,  100,   60],
       [ 300,   45,   80,   50],
       [ 700,  160,   60,   30],
       [ 500,  200,   50,   20],
       [ 200,   30,   50,   40],
       [ 100,   50,   20,   15],
       [ 800,  180,  200,  100],
       [1200,  300,  250,  115],
       [ 250,  100,   20,   25],
       [ 400,   80,   30,   10],
       [1000,  250,  130,  100],
       [ 300,   75,   60,   45]])

In [ ]:
Y

array([[200, 100,  90],
       [600, 250,  60],
       [600, 450,  40],
       [500, 360,  60],
       [330, 250,  50],
       [180,  75,  80],
       [500, 200, 100],
       [750, 500,  65],
       [350, 180,  50],
       [440, 230,  80],
       [300, 130, 100],
       [200,  80,  85],
       [160,  90, 100],
       [125,  50,  80],
       [700, 400,  90],
       [750, 400,  55],
       [180,  70, 100],
       [130,  60,  90],
       [600, 270,  95],
       [225, 100,  40]])

## 11. Comparando DEA CCR e DEA BCC para `dea_data_2.csv`

Agora vamos executar e comparar os resultados dos modelos CCR (retornos constantes de escala) e BCC (retornos variáveis de escala), ambos orientados a output, para o conjunto de dados `dea_data_2.csv`. Isso nos permitirá observar como a suposição de retornos de escala afeta os valores de eficiência.

In [ ]:
resp_ccr_o = dea.CCR_O(X, Y)
resp_bcc_o =  dea.BCC_O(X, Y)

In [ ]:
resp_ccr_o

[(array([0.     , 0.00341, 0.00732, 0.     , 0.00946, 0.     , 0.01888]),
  0.88169),
 (array([0.00167, 0.     , 0.     , 0.     , 0.00028, 0.00443, 0.00732]),
  0.59059),
 (array([0.     , 0.00222, 0.     , 0.00034, 0.     , 0.     , 0.00855]), 1.0),
 (array([0.     , 0.00278, 0.     , 0.00034, 0.0003 , 0.     , 0.01049]), 1.0),
 (array([0.     , 0.004  , 0.     , 0.0007 , 0.     , 0.00031, 0.01371]), 1.0),
 (array([0.0054 , 0.     , 0.00035, 0.     , 0.00099, 0.01358, 0.02546]),
  0.59913),
 (array([0.002  , 0.     , 0.     , 0.00022, 0.     , 0.00059, 0.01532]), 1.0),
 (array([0.000e+00, 2.000e-03, 0.000e+00, 3.000e-05, 0.000e+00, 0.000e+00,
         1.077e-02]),
  1.0),
 (array([0.00062, 0.00435, 0.     , 0.     , 0.     , 0.00621, 0.01553]),
  0.644),
 (array([0.00227, 0.     , 0.     , 0.00028, 0.     , 0.     , 0.01832]), 1.0),
 (array([0.00325, 0.     , 0.00024, 0.     , 0.00061, 0.00792, 0.01579]),
  0.95518),
 (array([0.005  , 0.     , 0.     , 0.00062, 0.     , 0.     , 0.04

In [ ]:
for dmu, ccr, bcc in zip(df.dmu,resp_ccr_o, resp_bcc_o):
    print("{:s} - Ef. Relativa CCR: {:.5f}- Ef. Relativa BCC: {:.5f}".format(dmu, ccr[1], bcc[1]))

dmu1 - Ef. Relativa CCR: 0.88169- Ef. Relativa BCC: 0.98184
dmu2 - Ef. Relativa CCR: 0.59059- Ef. Relativa BCC: 0.95455
dmu3 - Ef. Relativa CCR: 1.00000- Ef. Relativa BCC: 1.00000
dmu4 - Ef. Relativa CCR: 1.00000- Ef. Relativa BCC: 1.00000
dmu5 - Ef. Relativa CCR: 1.00000- Ef. Relativa BCC: 1.00000
dmu6 - Ef. Relativa CCR: 0.59913- Ef. Relativa BCC: 0.80000
dmu7 - Ef. Relativa CCR: 1.00000- Ef. Relativa BCC: 1.00000
dmu8 - Ef. Relativa CCR: 1.00000- Ef. Relativa BCC: 1.00000
dmu9 - Ef. Relativa CCR: 0.64400- Ef. Relativa BCC: 0.71317
dmu10 - Ef. Relativa CCR: 1.00000- Ef. Relativa BCC: 1.00000
dmu11 - Ef. Relativa CCR: 0.95518- Ef. Relativa BCC: 1.00000
dmu12 - Ef. Relativa CCR: 0.89710- Ef. Relativa BCC: 0.92661
dmu13 - Ef. Relativa CCR: 1.00000- Ef. Relativa BCC: 1.00000
dmu14 - Ef. Relativa CCR: 1.00000- Ef. Relativa BCC: 1.00000
dmu15 - Ef. Relativa CCR: 0.82683- Ef. Relativa BCC: 1.00000
dmu16 - Ef. Relativa CCR: 0.71040- Ef. Relativa BCC: 1.00000
dmu17 - Ef. Relativa CCR: 1.00000

In [ ]:
resp_bcc_o

[(array([0.0000e+00, 2.3300e-03, 8.5200e-03, 4.1000e-04, 1.1400e-03,
         0.0000e+00, 9.8200e-03, 5.5253e-01]),
  0.98184),
 (array([0.00167, 0.     , 0.     , 0.     , 0.     , 0.00238, 0.     ,
         0.69048]),
  0.95455),
 (array([0.     , 0.00222, 0.     , 0.00034, 0.     , 0.     , 0.00855,
         0.     ]),
  1.0),
 (array([0.     , 0.00278, 0.     , 0.00034, 0.0003 , 0.     , 0.01049,
         0.     ]),
  1.0),
 (array([0.     , 0.004  , 0.     , 0.0007 , 0.     , 0.00031, 0.01371,
         0.     ]),
  1.0),
 (array([-0.    ,  0.    ,  0.0125,  0.    ,  0.    , -0.    ,  0.    ,
          1.25  ]),
  0.8),
 (array([0.002  , 0.     , 0.     , 0.     , 0.     , 0.     , 0.02467,
         0.01333]),
  1.0),
 (array([0.   , 0.002, 0.   , 0.   , 0.   , 0.   , 0.011, 0.01 ]), 1.0),
 (array([0.00184, 0.00197, 0.     , 0.     , 0.     , 0.0063 , 0.00117,
         0.70189]),
  0.71317),
 (array([0.     , 0.00435, 0.     , 0.     , 0.     , 0.00256, 0.01528,
         0.03133]),